<a href="https://colab.research.google.com/github/gravity102424/ESAA/blob/main/ESAA_OB_week05_1_review_Medical_Cost_Personal_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Medical Cost Personal Datasets

https://www.kaggle.com/datasets/mirichoi0218/insurance

# 수상작 리뷰


---

### 1. 코드 흐름 요약

이 데이터셋을 활용한 예측 모델 코드는 일반적으로 다음 4단계의 흐름을 가집니다.

1. **데이터 로드 및 기본 파악:** `pandas`를 이용해 CSV 파일을 불러오고, 결측치(Missing Values)나 데이터 타입 등을 확인합니다. (이 데이터셋은 결측치가 없는 깔끔한 데이터입니다.)
2. **탐색적 데이터 분석(EDA):** `seaborn`과 `matplotlib`을 사용해 타겟 변수인 `charges`(의료비)의 분포를 확인합니다. 특히 `smoker`(흡연 여부)와 `bmi`가 의료비에 미치는 영향을 교차 시각화하여 직관적인 인사이트를 얻습니다.
3. **데이터 전처리(Preprocessing):** 모델이 학습할 수 있도록 문자열로 된 범주형 변수(`sex`, `smoker`, `region`)를 수치형으로 변환(원-핫 인코딩 등)합니다. 이후 데이터를 학습용(Train)과 테스트용(Test)으로 분리합니다.
4. **모델 학습 및 평가:** 다중 선형 회귀(Linear Regression)나 앙상블 모델(Random Forest, XGBoost 등)을 학습시킵니다. 이후 $R^2$ Score(결정계수)와 RMSE(평균 제곱근 오차) 지표를 통해 모델의 예측 성능을 평가합니다.

---

### 2. 새롭게 알게 된 내용

* **특정 변수의 압도적인 영향력:** 흡연 여부(`smoker`)가 의료비 청구 금액에 가장 절대적인 영향을 미친다는 점을 데이터를 통해 명확히 확인할 수 있습니다.
* **변수 간의 상호작용(Interaction):** 단순히 BMI가 높다고 의료비가 무조건 높은 것은 아니지만, **'흡연자이면서 동시에 BMI가 30 이상인 경우'** 의료비가 기하급수적으로 폭등한다는 데이터 패턴을 발견할 수 있습니다.
* **비대칭 데이터의 존재:** `charges` 데이터의 분포가 정규분포가 아니라 왼쪽으로 크게 치우친(Right-skewed) 형태를 띤다는 것을 알 수 있습니다.

---

### 3. 어려웠던 내용

* **범주형 데이터의 인코딩 방식 선택:** `region`(지역)이나 `sex`(성별) 같은 데이터를 0, 1, 2 숫자로 바꾸는 라벨 인코딩(Label Encoding)을 쓸지, 아니면 더미 변수로 만드는 원-핫 인코딩(One-Hot Encoding)을 쓸지 결정하고 적용하는 과정이 처음엔 헷갈릴 수 있습니다.
* **회귀 모델의 평가지표 해석:** 분류 문제의 '정확도(Accuracy)'와 달리, 회귀 문제에서는 $R^2$가 1에 가까울수록 좋고 RMSE는 낮을수록 좋다는 등 다양한 평가 지표의 의미를 정확히 이해하고 비교하는 부분이 까다롭습니다.

---

### 4. 배울 점

* **EDA(탐색적 데이터 분석)의 중요성:** 무작정 머신러닝 모델에 데이터를 밀어 넣기 전에, 그래프를 그려 데이터를 관찰하는 것만으로도 어떤 변수가 가장 중요한지(Feature Importance) 파악할 수 있다는 점을 배울 수 있습니다.
* **모델 성능을 높이는 전처리:** 로그 변환(Log Transformation)을 통해 치우친 `charges` 데이터를 정규분포에 가깝게 만들어 주면 선형 회귀 모델의 성능이 훨씬 좋아진다는 데이터 전처리의 기초를 배울 수 있습니다.

---

### 5. 주요 코드 정리

데이터 전처리부터 모델 학습까지의 핵심 파이썬 코드 스니펫입니다.

```python
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1. 데이터 로드
df = pd.read_csv('insurance.csv')

# 2. EDA (흡연 여부와 BMI가 의료비에 미치는 영향 시각화)
# 이 코드를 통해 흡연+고도비만 그룹의 의료비가 매우 높음을 시각적으로 확인 가능
sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df)

# 3. 데이터 전처리 (범주형 변수를 One-Hot Encoding으로 변환)
# drop_first=True를 통해 다중공선성(Multicollinearity) 방지
df_encoded = pd.get_dummies(df, drop_first=True)

# 4. 데이터 분리 (Train / Test)
X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. 모델 학습 및 평가
model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"R2 Score: {r2_score(y_test, predictions):.4f}")
```

---